In [11]:
import os
import numpy as np
import matplotlib.pyplot as plt

from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy

from week6_historical_env import HistoricalTradingEnv


def train():

    env = HistoricalTradingEnv()

    model = PPO(
        "MlpPolicy",
        env,
        learning_rate=3e-4,
        n_steps=128,
        batch_size=32,
        gamma=0.99,
        verbose=1
    )

    model.learn(
        total_timesteps=50000
    )

    os.makedirs(
        "models",
        exist_ok=True
    )

    model.save(
        "models/week6_ppo"
    )

    env.close()

    return model


def evaluate(
    model,
    episodes=5
):

    env = HistoricalTradingEnv()

    rewards = []

    pnl = []

    for ep in range(episodes):

        obs, info = env.reset()

        done = False

        total_reward = 0.0

        curve = [0.0]

        while not done:

            action, _ = model.predict(
                obs,
                deterministic=True
            )

            obs, reward, terminated, truncated, info = env.step(
                action
            )

            total_reward += reward

            curve.append(total_reward)

            done = terminated or truncated

        rewards.append(
            total_reward
        )

        pnl.append(
            curve
        )

        print(
            f"Episode {ep+1}: {total_reward:.4f}"
        )

    env.close()

    return rewards, pnl


def random_agent(
    episodes=5
):

    env = HistoricalTradingEnv()

    rewards = []

    for ep in range(episodes):

        obs, info = env.reset()

        done = False

        total = 0.0

        while not done:

            action = env.action_space.sample()

            obs, reward, terminated, truncated, info = env.step(
                action
            )

            total += reward

            done = terminated or truncated

        rewards.append(total)

    env.close()

    return rewards


def plot_curve(
    pnl
):

    os.makedirs(
        "plots",
        exist_ok=True
    )

    plt.figure(
        figsize=(8,4)
    )

    for curve in pnl:

        plt.plot(
            curve
        )

    plt.xlabel(
        "Step"
    )

    plt.ylabel(
        "Cumulative Reward"
    )

    plt.title(
        "Week 6 PPO Trading"
    )

    plt.tight_layout()

    plt.savefig(
        "plots/week6_pnl.png"
    )

    plt.close()


if __name__ == "__main__":

    model = train()

    env = HistoricalTradingEnv()

    mean_reward, std_reward = evaluate_policy(
        model,
        env,
        n_eval_episodes=5,
        deterministic=True
    )

    print()

    print(
        "Evaluation Mean:",
        round(mean_reward,4)
    )

    print(
        "Evaluation Std:",
        round(std_reward,4)
    )

    rewards, pnl = evaluate(
        model,
        episodes=5
    )

    random_rewards = random_agent(
        episodes=5
    )

    plot_curve(
        pnl
    )

    print()

    print(
        "PPO Mean Reward:",
        round(np.mean(rewards),4)
    )

    print(
        "Random Mean Reward:",
        round(np.mean(random_rewards),4)
    )

    print(
        "Best Episode:",
        round(np.max(rewards),4)
    )

    print(
        "Worst Episode:",
        round(np.min(rewards),4)
    )

    print()

    print(
        "Model saved in models/week6_ppo"
    )

    print(
        "Plot saved in plots/week6_pnl.png"
    )

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
----------------------------
| time/              |     |
|    fps             | 267 |
|    iterations      | 1   |
|    time_elapsed    | 0   |
|    total_timesteps | 128 |
----------------------------
------------------------------------------
| time/                   |              |
|    fps                  | 176          |
|    iterations           | 2            |
|    time_elapsed         | 1            |
|    total_timesteps      | 256          |
| train/                  |              |
|    approx_kl            | 0.0029587885 |
|    clip_fraction        | 0            |
|    clip_range           | 0.2          |
|    entropy_loss         | -1.1         |
|    explained_variance   | -0.0112      |
|    learning_rate        | 0.0003       |
|    loss                 | 2.14         |
|    n_updates            | 10           |
|    policy_gradient_loss | -0.00605     |
|    value_loss

C:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\stable_baselines3\common\evaluation.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(



Evaluation Mean: 77.7824
Evaluation Std: 0.0
Episode 1: 77.7824
Episode 2: 77.7824
Episode 3: 77.7824
Episode 4: 77.7824
Episode 5: 77.7824

PPO Mean Reward: 77.7824
Random Mean Reward: -129.8047
Best Episode: 77.7824
Worst Episode: 77.7824

Model saved in models/week6_ppo
Plot saved in plots/week6_pnl.png
